# RAG Ingestion Pipeline
Connect to Qdrant, initialize Google embedder, create collections.

In [ ]:
import os
import re
import uuid
from pathlib import Path
from abc import ABC, abstractmethod
from dataclasses import dataclass

from google import genai
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams

# Configuration
QDRANT_URL = os.environ.get("QDRANT_URL", "http://192.168.10.191:6333")
GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY", "")
EMBEDDING_MODEL = "text-embedding-004"
EMBEDDING_DIM = 768
ARTICLES_COLLECTION = "articles"
PODCASTS_COLLECTION = "podcast_episodes"

DATA_DIR = Path("../data")
ARTICLES_DIR = DATA_DIR / "articles"
PODCASTS_DIR = DATA_DIR / "podcasts"

# Clients
qdrant = QdrantClient(url=QDRANT_URL)
genai_client = genai.Client(api_key=GOOGLE_API_KEY)

# Create or recreate collections
for name in [ARTICLES_COLLECTION, PODCASTS_COLLECTION]:
    existing = [c.name for c in qdrant.get_collections().collections]
    if name in existing:
        qdrant.delete_collection(name)
        print(f"Deleted existing collection: {name}")
    qdrant.create_collection(
        collection_name=name,
        vectors_config=VectorParams(size=EMBEDDING_DIM, distance=Distance.COSINE),
    )
    print(f"Created collection: {name}")


def embed_texts(texts: list[str]) -> list[list[float]]:
    """Embed a batch of texts. Handles batching for large lists."""
    all_embeddings = []
    batch_size = 100  # Google API limit per request
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        response = genai_client.models.embed_content(
            model=EMBEDDING_MODEL,
            contents=batch,
        )
        all_embeddings.extend([e.values for e in response.embeddings])
    return all_embeddings

## Transcript Parser
Pluggable parser for podcast transcripts. Default implementation handles `[HH:MM:SS] Speaker: text` format.

In [ ]:
@dataclass
class TranscriptSegment:
    """A single parsed segment from a transcript."""
    timestamp: str  # "HH:MM:SS"
    seconds: int  # timestamp in total seconds
    speaker: str
    text: str


class TranscriptParser(ABC):
    """Base class for transcript parsers. Subclass to support different formats."""

    @abstractmethod
    def parse(self, content: str) -> list[TranscriptSegment]:
        ...


class TimestampSpeakerParser(TranscriptParser):
    """Parses transcripts in [HH:MM:SS] Speaker: text format."""

    PATTERN = re.compile(r"\[(\d{2}:\d{2}:\d{2})\]\s*([^:]+):\s*(.*)")

    def parse(self, content: str) -> list[TranscriptSegment]:
        segments = []
        for line in content.strip().splitlines():
            match = self.PATTERN.match(line.strip())
            if not match:
                continue
            timestamp, speaker, text = match.groups()
            h, m, s = map(int, timestamp.split(":"))
            segments.append(
                TranscriptSegment(
                    timestamp=timestamp,
                    seconds=h * 3600 + m * 60 + s,
                    speaker=speaker.strip(),
                    text=text.strip(),
                )
            )
        return segments


def build_sliding_windows(
    segments: list[TranscriptSegment],
    window_seconds: int = 30,
    overlap_seconds: int = 10,
) -> list[dict]:
    """Build sliding windows from parsed transcript segments."""
    if not segments:
        return []

    windows = []
    start_idx = 0

    while start_idx < len(segments):
        window_start = segments[start_idx].seconds
        window_end = window_start + window_seconds

        # Collect segments within this window
        window_segments = []
        for seg in segments[start_idx:]:
            if seg.seconds < window_end:
                window_segments.append(seg)
            else:
                break

        if not window_segments:
            start_idx += 1
            continue

        window_text = " ".join(
            f"{seg.speaker}: {seg.text}" for seg in window_segments
        )

        windows.append(
            {
                "timestamp_start": window_segments[0].timestamp,
                "timestamp_end": window_segments[-1].timestamp,
                "segment_texts": [seg.text for seg in window_segments],
                "window_text": window_text,
            }
        )

        # Advance by (window - overlap)
        advance_to = window_start + window_seconds - overlap_seconds
        new_start = start_idx
        for i, seg in enumerate(segments[start_idx:], start=start_idx):
            if seg.seconds >= advance_to:
                new_start = i
                break
        else:
            break  # No more segments to process

        if new_start == start_idx:
            start_idx += 1  # Ensure progress
        else:
            start_idx = new_start

    return windows


# Default parser
parser = TimestampSpeakerParser()

## Article Ingestion
Load articles from `data/articles/`, chunk semantically, embed, and upsert.

In [ ]:
def chunk_article(text: str, max_tokens: int = 500, min_tokens: int = 100) -> list[str]:
    """Split article text into semantic chunks by paragraphs, with size limits.

    Approximates tokens as words (rough but sufficient for chunking).
    """
    # Split on double newlines or markdown headings
    raw_chunks = re.split(r"\n\s*\n|(?=^#{1,3}\s)", text.strip(), flags=re.MULTILINE)
    raw_chunks = [c.strip() for c in raw_chunks if c.strip()]

    merged: list[str] = []
    buffer = ""

    for chunk in raw_chunks:
        word_count = len(chunk.split())

        if word_count > max_tokens:
            # Flush buffer first
            if buffer:
                merged.append(buffer.strip())
                buffer = ""
            # Split large chunk at sentence boundaries
            sentences = re.split(r"(?<=[.!?])\s+", chunk)
            sub_buffer = ""
            for sentence in sentences:
                if len((sub_buffer + " " + sentence).split()) > max_tokens and sub_buffer:
                    merged.append(sub_buffer.strip())
                    sub_buffer = sentence
                else:
                    sub_buffer = (sub_buffer + " " + sentence).strip()
            if sub_buffer:
                merged.append(sub_buffer.strip())
        elif buffer and len((buffer + "\n\n" + chunk).split()) > max_tokens:
            merged.append(buffer.strip())
            buffer = chunk
        else:
            buffer = (buffer + "\n\n" + chunk).strip() if buffer else chunk

    if buffer:
        merged.append(buffer.strip())

    # Merge small trailing chunks
    final: list[str] = []
    for chunk in merged:
        if final and len(chunk.split()) < min_tokens:
            final[-1] = final[-1] + "\n\n" + chunk
        else:
            final.append(chunk)

    return final


# Ingest articles
article_files = list(ARTICLES_DIR.glob("*.md")) + list(ARTICLES_DIR.glob("*.txt"))
print(f"Found {len(article_files)} article(s)")

all_article_points: list[PointStruct] = []

for filepath in article_files:
    title = filepath.stem.replace("-", " ").replace("_", " ").title()
    content = filepath.read_text()
    chunks = chunk_article(content)
    print(f"  {title}: {len(chunks)} chunk(s)")

    embeddings = embed_texts(chunks)

    for i, (chunk_text, embedding) in enumerate(zip(chunks, embeddings)):
        all_article_points.append(
            PointStruct(
                id=str(uuid.uuid4()),
                vector=embedding,
                payload={
                    "article_title": title,
                    "article_url": None,
                    "chunk_text": chunk_text,
                    "chunk_index": i,
                },
            )
        )

if all_article_points:
    qdrant.upsert(collection_name=ARTICLES_COLLECTION, points=all_article_points, wait=True)
    print(f"\nUpserted {len(all_article_points)} article point(s)")
else:
    print("\nNo articles to ingest")

## Podcast Ingestion
Load transcripts from `data/podcasts/`, parse, build sliding windows, embed, and upsert.

In [ ]:
podcast_files = list(PODCASTS_DIR.glob("*.txt"))
print(f"Found {len(podcast_files)} podcast transcript(s)")

all_podcast_points: list[PointStruct] = []

for filepath in podcast_files:
    episode_title = filepath.stem.replace("-", " ").replace("_", " ").title()
    episode_id = filepath.stem
    content = filepath.read_text()

    segments = parser.parse(content)
    windows = build_sliding_windows(segments)
    print(f"  {episode_title}: {len(segments)} segment(s) -> {len(windows)} window(s)")

    window_texts = [w["window_text"] for w in windows]
    if not window_texts:
        continue

    embeddings = embed_texts(window_texts)

    for window, embedding in zip(windows, embeddings):
        all_podcast_points.append(
            PointStruct(
                id=str(uuid.uuid4()),
                vector=embedding,
                payload={
                    "episode_title": episode_title,
                    "episode_id": episode_id,
                    "timestamp_start": window["timestamp_start"],
                    "timestamp_end": window["timestamp_end"],
                    "segment_texts": window["segment_texts"],
                    "window_text": window["window_text"],
                },
            )
        )

if all_podcast_points:
    qdrant.upsert(collection_name=PODCASTS_COLLECTION, points=all_podcast_points, wait=True)
    print(f"\nUpserted {len(all_podcast_points)} podcast point(s)")
else:
    print("\nNo podcasts to ingest")

## Verification
Check collection counts and run test queries.

In [ ]:
# Collection stats
for name in [ARTICLES_COLLECTION, PODCASTS_COLLECTION]:
    info = qdrant.get_collection(name)
    print(f"{name}: {info.points_count} points")

# Test article search
print("\n--- Article search: 'What is reinforcement learning?' ---")
test_query = embed_texts(["What is reinforcement learning?"])[0]
results = qdrant.query_points(
    collection_name=ARTICLES_COLLECTION,
    query=test_query,
    limit=3,
    with_payload=True,
)
for point in results.points:
    print(f"  [{point.score:.3f}] {point.payload['article_title']}: {point.payload['chunk_text'][:100]}...")

# Test podcast search
print("\n--- Podcast search: 'What is deep learning?' ---")
test_query = embed_texts(["What is deep learning?"])[0]
results = qdrant.query_points(
    collection_name=PODCASTS_COLLECTION,
    query=test_query,
    limit=3,
    with_payload=True,
)
for point in results.points:
    print(f"  [{point.score:.3f}] {point.payload['episode_title']} ({point.payload['timestamp_start']}): {point.payload['window_text'][:100]}...")